In [3]:
import pyspark                                                                                                   
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder\
    .master("local[*]")\
    .appName('test')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 21:05:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-08 18:09:17--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.161.127.198, 3.161.127.221, 3.161.127.152, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.161.127.198|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  2.04MB/s    in 95s     

2026-03-08 18:10:53 (3.11 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [14]:
df_initial = spark.read\
    .option("header","true")\
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [7]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

In [19]:
df.limit(100).write.mode("overwrite").parquet('head.parquet')

In [18]:
!wc -l head.parquet

wc: head.parquet: Is a directory
0 head.parquet


In [4]:
import pandas as pd

In [20]:
df_pandas = pd.read_parquet('head.parquet')

In [22]:
df_pandas.dtypes

hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object

In [28]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag',

<h3> This one below won't work with Parquets as they don't have types</h3>

In [8]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField('hvfhs_license_num',      T.StringType(),    True),
    T.StructField('dispatching_base_num',   T.StringType(),    True),
    T.StructField('originating_base_num',   T.StringType(),    True),
    T.StructField('request_datetime',       T.TimestampType(), True),
    T.StructField('on_scene_datetime',      T.TimestampType(), True),
    T.StructField('pickup_datetime',        T.TimestampType(), True),
    T.StructField('dropoff_datetime',       T.TimestampType(), True),
    T.StructField('PULocationID',           T.IntegerType(),   True),  # ~265 NYC zones, Long overkill
    T.StructField('DOLocationID',           T.IntegerType(),   True),  # same
    T.StructField('trip_miles',             T.DoubleType(),    True),
    T.StructField('trip_time',              T.IntegerType(),   True),  # seconds, never hits Long range
    T.StructField('base_passenger_fare',    T.DoubleType(),    True),
    T.StructField('tolls',                  T.DoubleType(),    True),
    T.StructField('bcf',                    T.DoubleType(),    True),
    T.StructField('sales_tax',              T.DoubleType(),    True),
    T.StructField('congestion_surcharge',   T.DoubleType(),    True),
    T.StructField('airport_fee',            T.DoubleType(),    True),
    T.StructField('tips',                   T.DoubleType(),    True),
    T.StructField('driver_pay',             T.DoubleType(),    True),
    T.StructField('shared_request_flag',    T.StringType(),    True),
    T.StructField('shared_match_flag',      T.StringType(),    True),
    T.StructField('access_a_ride_flag',     T.StringType(),    True),
    T.StructField('wav_request_flag',       T.StringType(),    True),
    T.StructField('wav_match_flag',         T.StringType(),    True),
])

In [ ]:
df = spark.read\
    .option("header","true")\
    .schema(schema)\
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [32]:
#You will need something like this:

In [9]:
df = spark.read \
    .option("header", "true") \
    .parquet('fhvhv_tripdata_2021-01.parquet')

df = df \
    .withColumn('PULocationID', df.PULocationID.cast(T.IntegerType())) \
    .withColumn('DOLocationID', df.DOLocationID.cast(T.IntegerType())) \
    .withColumn('trip_time',    df.trip_time.cast(T.IntegerType()))

In [10]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N'),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 45, 56), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DO

In [35]:
df.repartition(24)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp_ntz, on_scene_datetime: timestamp_ntz, pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int, trip_miles: double, trip_time: int, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string]

In [11]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: integer (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access

In [12]:
df.select('pickup_datetime','dropoff_datetime', 'PULocationID', 'DOLocationID')\
    .filter(df.hvfhs_license_num == 'HV0003')\
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|
|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|
|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|
|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|
|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|
|2021-01-01 00:14:30|2021-01-01 00:50:27|          71|         226|
|2021-01-01 00:22:54|2021-01-01 00:30:20|         112|         255|
|2021-01-01 00:40:12|2021-01-01 00:53:31|         255|         232|
|2021-01-01 00:56:45|2021-01-01 01:17:42|         232|         198|
|2021-01-01 00:29:04|2021-01-01 00:36:27|         113|          48|
|2021-01-01 00:48:56|2021-01-01 00:59:12|         239|          75|
|2021-01-01 00:11:53|2021-01-01 00:18:06|       

In [17]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [16]:
#user functions

In [21]:
def categorize_trip(seconds):
    if seconds is None:
        return 'unknown'
    elif seconds < 300:
        return 'short'
    elif seconds < 1800:
        return 'medium'
    elif seconds < 3600:
        return 'long'
    else:
        return 'very long'

In [19]:
categorize_trip_udf = F.udf(categorize_trip, StringType())

df.withColumn('trip_category', categorize_trip_udf(df.trip_time)) \
  .select('pickup_datetime', 'trip_miles', 'trip_time', 'trip_category', 'base_passenger_fare') \
  .show(10)

+-------------------+----------+---------+-------------+-------------------+
|    pickup_datetime|trip_miles|trip_time|trip_category|base_passenger_fare|
+-------------------+----------+---------+-------------+-------------------+
|2021-01-01 00:33:44|      5.26|      923|       medium|              22.28|
|2021-01-01 00:55:19|      3.65|     1382|       medium|              18.36|
|2021-01-01 00:23:56|      3.51|      849|       medium|              14.05|
|2021-01-01 00:42:51|      0.74|      179|        short|               7.91|
|2021-01-01 00:48:14|       9.2|     1228|       medium|              27.11|
|2021-01-01 00:06:59|     9.725|     2162|         long|              28.11|
|2021-01-01 00:50:00|     2.469|      897|       medium|              25.03|
|2021-01-01 00:14:30|     13.53|     2157|         long|              29.67|
|2021-01-01 00:22:54|       1.6|      446|       medium|               6.89|
|2021-01-01 00:40:12|       3.2|      800|       medium|              11.51|